In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import MemorySaver #Stores everything in RAM

In [5]:
from dotenv import load_dotenv

import os

load_dotenv()

True

In [6]:
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

In [8]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [7]:
def chat_node(state:ChatState):
    #Take user query from user
    messages=state['messages']
    #LLM
    response=llm.invoke(messages)
    #Return
    return{'messages':[response]}

In [9]:
checkpointer = MemorySaver() #saving memory in RAM
graph=StateGraph(ChatState)

#Nodes
graph.add_node('chat_node',chat_node)
#Edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [12]:
#Thread_id for different sessions and memory
thread_id='1'
while True:
    user_message=input('type here: ')
    print('Saurabh: ',user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config= {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke(

    {"messages": [HumanMessage(content=user_message)]},

    config=config

)
    print('Chatbot: ',response['messages'][-1].content)

Saurabh:  my name is saurabh
Chatbot:  Nice to meet you, Saurabh! How can I assist you today?
Saurabh:  tell mne what is paytm
Chatbot:  **Paytm** (short for **“Pay Through Mobile”**) is an Indian digital payments platform and financial services ecosystem that was launched in 2010 by One97 Communications. It started as a mobile wallet that let users store money digitally and make payments for a wide range of services, and has since expanded into a full‑featured app offering:

| Category | What It Does |
|----------|--------------|
| **Payments & Money Transfers** | QR‑code payments at merchants, bill payments (electricity, water, telecom, DTH, etc.), peer‑to‑peer transfers, and UPI (Unified Payments Interface) transactions. |
| **Financial Services** | Savings and fixed‑deposit accounts, mutual funds, insurance (health, life, motor), gold purchases, and personal loans. |
| **Commerce & Shopping** | An online marketplace for products (electronics, fashion, groceries, etc.), ticket booki

In [13]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='my name is saurabh', additional_kwargs={}, response_metadata={}, id='fa48e395-84f3-4d86-af29-26a6b3d97a7f'), AIMessage(content='Nice to meet you, Saurabh! How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "my name is saurabh". Probably they want some acknowledgement. We can respond politely, ask if they need help. No special instructions. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 76, 'total_tokens': 142, 'completion_time': 0.137178542, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.002868528, 'prompt_tokens_details': None, 'queue_time': 0.389570945, 'total_time': 0.14004707}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a06a93-4be5-7a40-a70e-9207cc5078